# Notebook 03: Build Text CSVs (base + small)

Creates:
- `outputs/features/all_texts_base.csv`
- `outputs/features/all_texts_small.csv`

Each row:
- `file` (stem)
- `label` (`ad` or `cn`)
- `text` (ASR transcript)

This is the dataset consumed by TF-IDF models.

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
FEATURE_DIR = PROJECT_ROOT / "outputs" / "features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

# You can keep using your existing features.csv as the authoritative file list/labels.
FEATURES_PATH = FEATURE_DIR / "features.csv"
assert FEATURES_PATH.exists(), f"Missing {FEATURES_PATH}. If you don't have it, create it from your pipeline first."

ASR_BASE_AD = PROJECT_ROOT / "outputs" / "asr" / "ad"
ASR_BASE_CN = PROJECT_ROOT / "outputs" / "asr" / "cn"
ASR_SMALL_AD = PROJECT_ROOT / "outputs" / "asr_small" / "ad"
ASR_SMALL_CN = PROJECT_ROOT / "outputs" / "asr_small" / "cn"

def read_txt(folder: Path, stem: str) -> str:
    p = folder / f"{stem}.txt"
    try:
        return p.read_text(encoding="utf-8", errors="ignore").strip()
    except Exception:
        return ""

df = pd.read_csv(FEATURES_PATH)
df = df[["file","label"]].copy()

def build(asr_ad: Path, asr_cn: Path, out_path: Path):
    texts = []
    for stem, lab in zip(df["file"], df["label"]):
        folder = asr_ad if lab == "ad" else asr_cn
        texts.append(read_txt(folder, stem))
    out = df.copy()
    out["text"] = texts
    out.to_csv(out_path, index=False)
    print(f"✅ Wrote {len(out)} rows to {out_path}")
    print(out["label"].value_counts())
    print("Empty transcripts:", int((out["text"].str.len()==0).sum()))

build(ASR_BASE_AD, ASR_BASE_CN, FEATURE_DIR/"all_texts_base.csv")
build(ASR_SMALL_AD, ASR_SMALL_CN, FEATURE_DIR/"all_texts_small.csv")